Neural Network: Binary to Integer Prediction

A neural network that takes a binary number as input (e.g., [1, 0, 1]) and predicts its integer value (e.g., 5).

What it learns
The mapping is deterministic: each bit has a fixed positional weight (2⁰, 2¹, 2²...). The network must approximate this function through training.

I/O structure
Input	Output
[0, 1, 0, 1]	5
[1, 0, 0, 0]	8
Why it's a good learning example
Simple, verifiable ground truth
Regression problem (predicting a number, not a class)
Forces the network to learn positional weighting implicitly


In [0]:
# Bulding the transformers, these will be used to convert the volume name to a binary number

def transformer_bin(num:int)->list:
    '''
    Converting the integer number to a binary number
    '''
    return [int(b) for b in bin(num)[2:].zfill(7)]

def tranformer_int(binario:list[int])->int:
    '''
    Converting the binary number to a integer
    '''
    return int(''.join(str(b) for b in binario),2)

x = transformer_bin(50)
print(x)
print(tranformer_int(x))

In [0]:
# Linear layer (z = Wx + b)
import numpy as np

def linear_layer(x:list[int],w:list[int],b:list[int])->list[int]:
    return np.dot(w, x) + b

x = transformer_bin(50)
w1 = np.random.randn(4,7) * 0.01
b1 = np.zeros(4)
linear_layer(x,w1,b1)

In [0]:
# Function to inizialize the parameters (weights, biases) and set the size of the layers
import numpy as np

def init_params(layer_size:int)->list[int]:
    params = []

    for i in range(len(layer_size) -1):
        w = np.random.randn(layer_size[i+1],layer_size[i]) * 0.01
        b = np.zeros((layer_size[i+1]))
        params.append([w,b])

    return params

print(init_params([7,8,2,1]))#Number of layers, number of neurons in each layer, number of neurons in the output layer, number of neurons in the output layer

In [0]:
# Activation functions, these will be used to convert the output of the linear layer to a value between 0 and 1

def sigmoide(output:list[int])->list[int]:
    return 1/(1 + np.exp(-output))

def relu(output:list[int])->list[int]:
    return np.maximum(output,0)



In [0]:
# Forward Propagation, this will be used to calculate the output of the network

def forward_prop(x:list[int],params:list[int]):
    z = []
    a = [x]

    for i in range(len(params)-1):
        z.append(linear_layer(a[i],params[i][0], params[i][1]))
        a.append(relu(z[i]))
    # ultima capa de activacion
    z.append(linear_layer(a[-1], params[-1][0],params[-1][1]))

    return z,a


In [0]:
# Calculating the loss function

def loss_mse(y_true:list[int],y_pred:list[int])->float:
    l =  np.mean((y_pred - y_true) ** 2)
    dl = 2 * (y_pred - y_true)

    return l, dl

In [0]:
# Back Propagation, this will be used to calculate the gradient of the loss function

def back_prop(a:list[int],z:list[int],params:list[int],y_true:list[int],x_original:list[int],dl:list[int]):

    '''
    Computes gradients for each layer using backpropagation

    parameters:
    a - layer activation, stored during forward propagation
    z - linear outputs per layer, stored during forward propagation
    params - weights and biases per layer
    y_true - real value
    x_original - original input (binary)
    dl - derivate of the MSE loss, starting point of backprop
    '''

    gradients = []

    for i in reversed(range(len(params))):  
        # si es la ultima capa, el error es directo, de lo contrario propago el error hacia atras 
        if i == len(params) -1:
            dz = dl
        else:
            #dz = np.dot(params[i+1][0].T, dz) * (a[i+1] * (1 - a[i+1]))

            dz = np.dot(params[i+1][0].T, dz) * (z[i] > 0).astype(float)

        a_prev = a[i] if i > 0 else x_original
        dw = np.outer(dz,a_prev)
        db = np.sum(dz)

        gradients.append([dw, db])

    return gradients[::-1] # se invierte para que quede en orden


In [0]:
def des_gradiente(lr:float,params:list[int],derivada:list[int]):
    '''
    Update weights and biases using gradient descent

    parameters:
    lr - learnign rate, control the size of each updated step
    params - current weights, biase
    gradients - gradient cumputed by the backpropagation
    '''
    for i in range(len(params)):
        params[i][0] = params[i][0] -lr  * derivada[i][0]
        params[i][1] = params[i][1] - lr *  derivada[i][1]
    
    return params

In [0]:
# Creacion de la funcion de epochs, para el loop de entrenamiento

def epoch(times:int,x_original:list[int],y_true:list[int],params:list[int],lr:float,):

    '''
    Main training loop, runs the forward propagation, computes, loss and updates the weights

    parameters:
    times - maximum number of training iterations
    x_train - list of the binary input vectors
    y_train - list of the expected ouput values
    params - initialized weights and biases 
    lr - learning rate

    Returns:
        params - trained weights and biases

    Flow per epoch:
        1. Forward prop - get prediction
        2. MSE loss - compute error and its derivative
        3. Backprop - compute gradients
        4. Gradient desc - update weights
    '''

    mejor_loss = float("inf")
    sin_mejora = 0

    for i in range(times):
        for j in range(len(x_original)):
            z,a = forward_prop(x_original[j],params)
            l, dl = loss_mse(y_true[j],z[-1])
            grad = back_prop(a,z,params,y_true[j],x_original[j],dl)
            #print(type(grad), len(grad), type(grad[0]))
            params = des_gradiente(lr, params, grad)

        if i % 100 == 0:
            print(f"epoch {i+1} - loss: {l}")
    return params 

In [0]:
import random

random.seed(42)
np.random.seed(42)

nums = list(range(99))
random.shuffle(nums) # shuffle to randomize the train test split

train = nums[:80] #80% for trainig
test_set = nums[80:] #20% for testing

x_train = [transformer_bin(i) for i in train]
y_train = train

x_test = [transformer_bin(i) for i in test_set]
y_test = test_set

params = init_params([7, 16, 1]) #architecture 7 inputs - 16 hidden neurons - 1 output neuron
params = epoch(4000, x_train, y_train, params, 0.0001)
params

In [0]:
# Model evaluation

correctos = 0 # the exact number of correct predictions
cercanos = 0 # the number of predictions within 2 units of the actual value

for i in range(len(x_test)):
    z, a = forward_prop(x_test[i], params)
    pred = round(z[-1][0])
    print(f"predicho: {pred}, real: {y_test[i]}")
    if pred == y_test[i]:
        correctos += 1
    if abs(pred - y_test[i]) <= 2:
        cercanos += 1

print(f"exactos: {correctos/len(x_test):.2%}")
print(f"dentro de ±2: {cercanos/len(x_test):.2%}")

In [0]:
# Test with different numbers out of the training range to see how the model behaves

for num in [100, 104, 105, 110, 115, 120, 127]:
    test = transformer_bin(num)
    z, a = forward_prop(test, params)
    print(f"predicho: {z[-1][0]:.1f}, real: {num}")